In [ ]:
# @title 1. Install Dependencies
# Installs llama-cpp-python with CUDA 12 support (Pre-built wheel for speed)
# Installs FastAPI server stack
!pip install -q llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!pip install -q fastapi uvicorn pyngrok nest_asyncio pydantic huggingface_hub

print("✅ Dependencies installed. Ready for Inference.")

: 

In [ ]:
# @title 2. Load Model (16k Context)
import os
import json
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
# We use the Instruct version for best instruction following
REPO_ID = "Qwen/Qwen2.5-Coder-14B-Instruct-GGUF"
FILENAME = "qwen2.5-coder-14b-instruct-q4_k_m.gguf"

# CONTEXT WINDOW SETTING
# 32k allows analyzing very large files.
CONTEXT_WINDOW = 16384

# ==========================================
# 🧠 LOAD ENGINE
# ==========================================
print(f"⬇️ Downloading {FILENAME} (This happens only once)...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print(f"✅ Model located at: {model_path}")

print(f"🔄 Initializing Lumen Engine with {CONTEXT_WINDOW} token context...")
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,        # Offload ALL layers to GPU
    n_ctx=CONTEXT_WINDOW,   # Context
    n_batch=256,            # Batch size for prompt processing
    verbose=False           # Reduce log spam
)
print("✅ Lumen Engine Online & Ready.")

# ==========================================
# 📝 DATA MODELS
# ==========================================
class AnalysisRequest(BaseModel):
    code: str
    language: str = "auto"
    framework: Optional[str] = None

# ==========================================
# 🛡️ THE "STRICT STANDARDS" PROMPT
# ==========================================
SYSTEM_PROMPT = """You are in the backend of "Lumen": a Principal Security Architect and Code Quality Auditor software.
Your goal is to perform a deep, comprehensive static analysis of the provided source code and then output in a STRICT JSON format.
Your supported programming languages are: C, C++, Java, Javascript, TypeScript, Python, SQL, Shell, Bash. If the input programming language in the prompt is not present in your supported programming language list then ONLY output: '{
  "summary": "Unsupported Language",
  "language_detected": "The programming language detected OR 'None'."}'

### 1. STANDARDS COMPLIANCE (STRICT)
You must analyze the code against the following specific industry standards. Strictly cite the specific rule or ID whenever a violation is found.

SECURITY STANDARDS:
OWASP Top 10 (2021).
SANS Top 25.
CWE Top 25 (2023): You MUST map every vulnerability to its exact *CWE_ID*.
CERT Secure Coding (C/C++/Java).
MISRA C/C++ (Safety Critical).
Data Protection: Identify violations of GDPR/HIPAA.
Secure Deserialization: Flag unsafe deserialization in Java, Python (pickle), and JavaScript.

QUALITY & MAINTAINABILITY:
ISO/IEC 25010: Analyze for Maintainability (modularity, reusability) and Reliability (error handling) and others.
SOLID Principles: Flag violations of SRP, OCP, LSP, ISP, and DIP.
Code Smells: Identify Cognitive Complexity, Spaghetti Code, Magic Numbers, and Dead Code and other major code smells.

Language Best Practices:
Python: PEP 8 & Idiomatic Python.
JS/TS: Airbnb Style & Async/Await best practices.
C++: C++ Core Guidelines & Modern C++ (RAII).
Java: Oracle Conventions & Effective Java patterns.

Also identify framework specific standard violations.

### 2. FALSE POSITIVE REDUCTION (CRITICAL)
*   Minimize the number of false positives in your analysis.
*   Context Matters!: Do not flag hardcoded credentials if the variable name indicates it is a generic placeholder (e.g., "example_key").
*   Test Code: If the code appears to be a unit test, suppress strict security warnings unless they involve dangerous patterns like `eval()`.
*   Taint Verification: Before flagging an Injection vulnerability, verify if the User Input actually reaches the Sink (Database/Shell). If input is sanitized, lower the severity.

### 3. ANALYSIS METHODOLOGY
*   Taint Analysis: Trace user input from entry points (API routes, CLI args) to sensitive sinks (Database, Eval, Shell, Logs).
*   Control Flow Analysis: Identify unreachable code, infinite loops, and improper error handling.
*   Data Flow Analysis: Identify uninitialized variables, resource leaks, and use-after-free errors.

### 4. OUTPUT FORMAT (STRICT JSON)
Output ONLY a valid, minified JSON object based on this example schema. Do not output markdown.

{
  "summary": "Executive summary of the code security and quality status.",
  "languages_detected": [
    "Python",
    "Javascript",
    "etc"
  ],
  "frameworks_detected": [
    "Flask",
    "React.js",
    "Node.js",
    "etc"
  ],
  "security_issues": [
    {
      "standard": "CWE-134",
      "title": "Uncontrolled format string",
      "description": "Technical description of the exploit and impact.",
      "start_line": 0,
      "end_line": 0,
      "snippet": "The exact problematic code snippet between the start_line and end_line, with \n",
      "severity": "Critical|High|Medium|Low",
      "confidence": "Certain|Likely|Potential",
      "remediation": "High-level brief remediation strategy.",
      "fixed_snippet": "The exact fixed code snippet that resolves the issue."
    },
    {
      "standard": "OWASP A03:2021",
      "title": "Injection via interpreted format string",
      "description": "Technical description of the exploit and impact.",
      "start_line": 0,
      "end_line": 0,
      "snippet": "The exact problematic code snippet between the start_line and end_line, with \n",
      "severity": "Critical|High|Medium|Low",
      "confidence": "Certain|Likely|Potential",
      "remediation": "High-level brief remediation strategy.",
      "fixed_snippet": "The exact fixed code snippet that resolves the issue."
    }
  ],
  "quality_issues": [
    {
      "standard": "PEP 8",
      "title": "Line exceeds maximum recommended length",
      "description": "Technical description of the cause and impact.",
      "start_line": 0,
      "end_line": 0,
      "snippet": "The exact problematic code snippet between the start_line and end_line, with \n",
      "severity": "Low|Medium|High",
      "type": "Maintainability|...",
      "remediation": "Refactor the line to improve readability and comply with style guidelines.",
      "fixed_snippet": "The exact fixed code snippet that resolves the issue."
    },
    {
      "standard": "ISO/IEC 25010",
      "title": "Function has multiple responsibilities",
      "description": "Technical description of the cause and impact.",
      "start_line": 0,
      "end_line": 0,
      "snippet": "The exact problematic code snippet between the start_line and end_line, with \n",
      "severity": "Low|Medium|High",
      "type": "Maintainability|...",
      "remediation": "Split the function into smaller, single-purpose functions.",
      "fixed_snippet": "The exact fixed code snippet that resolves the issue."
    }
  ],
  "business_logic_issues": [
    {
        "description": "Business level, non standard or architectural issue"
    },
    {
        "description": "Business level, non standard or architectural issue"
    }
  ]
}

Note: Duplicates are allowed in the security_issues and quality_issues and business_logic_issues arrays but they must have different locations.
"""

# ==========================================
# 🚀 FASTAPI APP
# ==========================================
app = FastAPI(title="Lumen AI Engine (Enterprise Standards)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def health_check():
    return {
        "status": "running",
        "model": "Qwen 2.5 14B Instruct",
        "context_window": CONTEXT_WINDOW,
        "mode": "Strict Standards Compliance"
    }

@app.post("/analyze")
def analyze_code(request: AnalysisRequest):
    # Construct the dynamic user prompt
    user_context = f"Language: {request.language}\n"
    if request.framework:
        user_context += f"Framework Context: {request.framework}\n"

    final_user_message = f"{user_context}\nCODE TO ANALYZE:\n{request.code}"

    try:
        # Run Inference
        response = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": final_user_message}
            ],
            response_format={"type": "json_object"}, # Force Valid JSON
            temperature=0.1, # Low temperature for factual accuracy
            max_tokens=4096, # Allow long responses for detailed reports
            top_p=0.95
        )

        # Extract and Parse
        content = response["choices"][0]["message"]["content"]
        return {"status": "success", "analysis": json.loads(content)}

    except Exception as e:
        print(f"❌ Error: {e}")
        # Return a structured error so frontend handles it gracefully
        return {
            "status": "error",
            "detail": str(e),
            "analysis": {
                "summary": "Analysis failed due to server error.",
                "security_issues": [],
                "quality_issues": []
            }
        }

In [ ]:
# @title 3. Start Server with Ngrok
import uvicorn
from pyngrok import ngrok
import nest_asyncio

# ==========================================
# 🔑 AUTHENTICATION
# ==========================================
NGROK_AUTH_TOKEN = "37LnlecjiNaYMFXQBHa3YONGyiU_3zrF8XD3PxCRMyWi9w7oL" 

if NGROK_AUTH_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    raise ValueError("❌ Please paste your Ngrok Auth Token!")

# 1. Setup Tunnel
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
# Force 127.0.0.1 to avoid IPv6 issues on Colab
tunnel = ngrok.connect("127.0.0.1:8000", "http")

print("\n" + "="*60)
print(f"🚀 LUMEN ENTERPRISE AI IS LIVE!")
print(f"🌍 API URL: {tunnel.public_url}")
print("="*60 + "\n")

# 2. Run Server (Async)
nest_asyncio.apply()
config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)
await server.serve()